In [ ]:
USUARIO = "IZARO" # @param ["IZARO","ENRIQUE"]
import os
import numpy as np
from datetime import datetime
from google.colab import drive
import pandas as pd
import matplotlib.pyplot as plt


rutas = {
    "IZARO": "/content/drive/MyDrive/05. PFG/2026 - TFG Izaro Fuzhi/Codigo",
    "ENRIQUE": "/content/drive/MyDrive/2-DOCENCIA/ALL Trabajos Fin de Estudios/2026 - TFG Izaro Fuzhi/Codigo"
}


DIRECTORIO = rutas[USUARIO]
drive.mount('/content/drive')
os.chdir(DIRECTORIO)  # ajusta la ruta

# Empezar a analizar

In [ ]:
# @title Configuración del fallback por LLM (detección de métrica)
# ─────────────────────────────────────────────────────────────────────────────
# Mecanismo de robustez: solo se usa si la heurística NO identifica ninguna
# métrica clara y quedan columnas ambiguas. En el dataset de tráfico de Madrid
# la heurística resuelve sola y esta rama no se ejecuta. Su valor está en la
# generalización a datasets arbitrarios con nombres de columna crípticos.
#
# DISEÑO: el proveedor de LLM está aislado en una única función
# `_llamar_llm(prompt) -> str`. Para cambiar de proveedor (Anthropic, OpenAI,
# un modelo local, etc.) solo se reescribe esa función; el resto del pipeline
# de detección es agnóstico al proveedor.
#
# PRIVACIDAD: NUNCA se envían filas de datos. Solo nombres de columna y
# estadísticos agregados (dtype, nº únicos, min, max, cv).
# ─────────────────────────────────────────────────────────────────────────────
USAR_LLM_FALLBACK = True   # @param {"type":"boolean"}

# Proveedor activo. Cambia este valor y/o reimplementa _llamar_llm más abajo.
PROVEEDOR_LLM = "gemini"  # @param ["gemini","anthropic","openai","ninguno"]

import os
try:
    from google.colab import userdata
    _LLM_API_KEY = (
      userdata.get('GEMINI_API_KEY')    if PROVEEDOR_LLM == "gemini"    else
      userdata.get('ANTHROPIC_API_KEY') if PROVEEDOR_LLM == "anthropic" else
      userdata.get('OPENAI_API_KEY')    if PROVEEDOR_LLM == "openai"    else None
  )
except Exception:
    _LLM_API_KEY = os.environ.get('ANTHROPIC_API_KEY' if PROVEEDOR_LLM == "anthropic"
                                  else 'OPENAI_API_KEY' if PROVEEDOR_LLM == "openai"
                                  else '')

# Modelo por proveedor (ajustable sin tocar la lógica)
_MODELO_LLM = {
    "gemini":    "gemini-2.5-flash",
    "anthropic": "claude-opus-4-5",
    "openai":    "gpt-4o",
    "ninguno":   None,
}.get(PROVEEDOR_LLM)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CAPA DE PROVEEDOR (lo único que cambia entre proveedores)
# ─────────────────────────────────────────────────────────────────────────────
def _llamar_llm(prompt, proveedor=None, modelo=None, api_key=None):
    """
    Interfaz única de LLM. Recibe un prompt (str), devuelve texto (str)
    o None si falla. Toda la dependencia de proveedor está AQUÍ y solo aquí.

    Para añadir un proveedor nuevo: añade una rama. El resto del código
    (perfilado, parseo, degradación) no se entera de qué proveedor es.
    """
    proveedor = proveedor or PROVEEDOR_LLM
    modelo    = modelo    or _MODELO_LLM
    api_key   = api_key   or _LLM_API_KEY

    if proveedor == "ninguno" or not api_key:
        return None

    try:
        if proveedor == "anthropic":
            import anthropic
            client = anthropic.Anthropic(api_key=api_key)
            resp = client.messages.create(
                model=modelo, max_tokens=500,
                messages=[{"role": "user", "content": prompt}],
            )
            return "".join(b.text for b in resp.content if b.type == "text")

        elif proveedor == "openai":
            from openai import OpenAI
            client = OpenAI(api_key=api_key)
            resp = client.chat.completions.create(
                model=modelo, max_tokens=500,
                messages=[{"role": "user", "content": prompt}],
            )
            return resp.choices[0].message.content
        elif proveedor == "gemini":
            import requests as _req, json as _json
            url = (
                f"https://generativelanguage.googleapis.com/v1beta/models/"
                f"{modelo}:generateContent?key={api_key}"
            )
            resp = _req.post(
                url,
                json={"contents": [{"parts": [{"text": prompt}]}]},
                timeout=15,
            )
            resp.raise_for_status()
            texto = (resp.json()["candidates"][0]["content"]["parts"][0]["text"])
            # Gemini a veces envuelve la respuesta en ```json ... ```
            return texto.strip().removeprefix("```json").removesuffix("```").strip()

        else:
            print(f"  (LLM) Proveedor no soportado: {proveedor!r}")
            return None

    except ImportError as e:
        print(f"  (LLM) Paquete del proveedor no instalado: {e}")
        return None
    except Exception as e:
        print(f"  (LLM) Error en la llamada ({type(e).__name__}): {e}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# LÓGICA AGNÓSTICA AL PROVEEDOR
# ─────────────────────────────────────────────────────────────────────────────
def _perfil_columnas_para_llm(df, var_tiempo, columnas):
    """Perfil AGREGADO de columnas. NO incluye datos crudos."""
    perfil = []
    N = len(df)
    for col in columnas:
        s = df[col]
        d = {"columna": col, "dtype": str(s.dtype), "n_unicos": int(s.nunique())}
        if np.issubdtype(s.dtype, np.number):
            d.update({
                "min": float(s.min()), "max": float(s.max()),
                "media": round(float(s.mean()), 4),
                "cv": round(float(s.std() / (abs(s.mean()) + 1e-9)), 4),
                "pct_unicos": round(s.nunique() / N, 4),
            })
        perfil.append(d)
    return perfil


def _detectar_metrica_via_llm(df, var_tiempo, candidatas):
    """
    Usa _llamar_llm (agnóstico al proveedor) para clasificar columnas.
    Devuelve lista de nombres de columna, o None si falla / no disponible.
    Degrada con elegancia: cualquier problema → None.
    """
    if not USAR_LLM_FALLBACK:
        return None

    import json as _json
    perfil = _perfil_columnas_para_llm(df, var_tiempo, candidatas)
    prompt = (
        "Eres un asistente que clasifica columnas de un dataset temporal.\n"
        "Dada esta lista de columnas candidatas con sus estadísticos agregados "
        "(NO se incluyen datos crudos), identifica cuáles representan MÉTRICAS "
        "cuantitativas analizables (magnitudes que varían en el tiempo y tiene "
        "sentido resumir: intensidad, ocupación, consumo, temperatura, etc.) "
        "y cuáles NO lo son (identificadores, códigos, coordenadas, categorías).\n\n"
        f"Variable temporal del dataset: {var_tiempo!r}\n"
        f"Columnas candidatas:\n{_json.dumps(perfil, ensure_ascii=False, indent=2)}\n\n"
        "Responde EXCLUSIVAMENTE con un objeto JSON, sin texto adicional, "
        'con esta forma: {"metricas": ["col1", "col2"], "razonamiento": "breve"}'
    )

    texto = _llamar_llm(prompt)
    if texto is None:
        print("  (LLM) No disponible → se omite el fallback por LLM.")
        return None

    try:
        texto = texto.strip().replace("```json", "").replace("```", "").strip()
        data = _json.loads(texto)
        metricas = [c for c in data.get("metricas", []) if c in candidatas]
        razon = data.get("razonamiento", "")
        if metricas:
            print(f"  (LLM) Métricas detectadas: {metricas}")
            print(f"  (LLM) Razonamiento: {razon}")
            return metricas
        print("  (LLM) No identificó métricas válidas entre las candidatas.")
        return None
    except (ValueError, KeyError) as e:
        print(f"  (LLM) Respuesta no parseable ({type(e).__name__}). Fallback manual.")
        return None

In [ ]:
#FICHERO = "data/6823.csv" # @param ["data/3600.csv","data/3730.csv","data/4301.csv","data/6791.csv","data/6822.csv","data/6823.csv"]
#FICHERO = "data/KwhConsumptionBlower78_1.csv"
#VAR_TIEMPO = "fecha"
#VAR_TIEMPO = "TxnDate"
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data")

print("Path to dataset files:", path)

FICHERO = os.path.join(path, "DailyDelhiClimateTrain.csv")
VAR_TIEMPO = None

# ─────────────────────────────────────────────────────────────────────────────
# DETECCIÓN AUTOMÁTICA DE VARIABLE MÉTRICA
# ─────────────────────────────────────────────────────────────────────────────
# Estrategia:
#   1. Heurística (reglas estadísticas + nombres): filtra columnas obviamente
#      no-métricas (coordenadas, IDs, categorías constantes, texto).
#   2. Si quedan candidatas ambiguas, llama a Claude API para desambiguar
#      con contexto semántico.
#   3. Si la API no está disponible, presenta la lista al usuario.
#
# El usuario siempre puede sobreescribir el resultado con VAR_METRICA_OVERRIDE.
# ─────────────────────────────────────────────────────────────────────────────

import re, json
import numpy as np

# Override manual (dejar en None para detección automática)
VAR_METRICA_OVERRIDE = None   # @param {"type":"raw"}

# ── Paso 0: cargar CSV sin filtrar columnas ───────────────────────────────
_df_raw = pd.read_csv(FICHERO)
_cols_all = list(_df_raw.columns)

# ── Detección automática de VAR_TIEMPO ───────────────────────────────────────
# Busca columnas temporales: datetime, o fecha+hora separadas para combinar.
# El usuario puede fijar VAR_TIEMPO manualmente si la detección falla.

VAR_TIEMPO_OVERRIDE = None  # @param {"type":"raw"}  — dejar None para autodetección


def _detectar_var_tiempo(df_raw):
    import pandas as pd

    _TOKENS_FECHA = {'fecha', 'date', 'day', 'dia', 'txndate', 'fec'}
    _TOKENS_HORA  = {'hora', 'time', 'hour', 'txntime', 'hh', 'hhmm'}

    def _tokenizar_col(col):
        return set(re.split(r'[_\-\s]+', col.lower())) | {col.lower()}

    cols_obj = [c for c in df_raw.columns if df_raw[c].dtype == object]
    cols_num = [c for c in df_raw.columns if pd.api.types.is_numeric_dtype(df_raw[c])]

    # Estrategia 1: columna datetime directa
    _candidata_solo_fecha = None
    for col in df_raw.columns:
        if df_raw[col].dtype != object:
            continue
        try:
            parsed = pd.to_datetime(df_raw[col], dayfirst=True, errors='coerce')
            if parsed.isna().mean() > 0.1:     # >10% sin parsear → no es fecha
                continue

            tiene_variabilidad_hora  = parsed.dt.hour.std() > 0 or parsed.dt.minute.std() > 0
            tiene_variabilidad_fecha = parsed.dt.date.nunique() > 1

            if tiene_variabilidad_hora and tiene_variabilidad_fecha:
                print(f"  (tiempo) Columna datetime detectada: {col!r}")
                return col, df_raw
            elif tiene_variabilidad_fecha and _candidata_solo_fecha is None:
                _candidata_solo_fecha = col
        except Exception:
            pass

    # Estrategia 2: par fecha + hora separadas
    cols_fecha = [c for c in cols_obj if _tokenizar_col(c) & _TOKENS_FECHA]
    cols_hora  = [c for c in cols_obj if _tokenizar_col(c) & _TOKENS_HORA]

    if cols_fecha and cols_hora:
        col_f, col_h = cols_fecha[0], cols_hora[0]
        try:
            combinada = pd.to_datetime(
                df_raw[col_f].astype(str) + ' ' + df_raw[col_h].astype(str)
            )
            df_raw = df_raw.copy()
            df_raw['_datetime'] = combinada
            print(f"  (tiempo) Fecha+hora combinadas: {col_f!r} + {col_h!r} → '_datetime'")
            print(f"           Rango: {combinada.min()} → {combinada.max()}")
            return '_datetime', df_raw
        except Exception as e:
            print(f"  (tiempo) Combinación fallida ({e})")

    # Estrategia 3: unix timestamp numérico
    for col in cols_num:
        tokens = _tokenizar_col(col)
        if tokens & {'timestamp', 'ts', 'epoch', 'unix', 'time'}:
            try:
                parsed = pd.to_datetime(df_raw[col], unit='s', errors='coerce')
                if parsed.notna().mean() > 0.9:
                    print(f"  (tiempo) Unix timestamp detectado: {col!r}")
                    df_raw = df_raw.copy()
                    df_raw['_datetime'] = parsed
                    return '_datetime', df_raw
            except Exception:
                pass

    # Usar candidata de solo fecha si no se encontró nada mejor  ← NUEVO
    if _candidata_solo_fecha is not None:
        print(f"  (tiempo) Columna de fecha detectada (sin hora): {_candidata_solo_fecha!r}")
        return _candidata_solo_fecha, df_raw

    # Fallback: primera columna parseable como fecha
    for col in cols_obj:
        try:
            pd.to_datetime(df_raw[col])
            print(f"  (tiempo) Fallback — usando primera columna fecha: {col!r}")
            return col, df_raw
        except Exception:
            pass

    print("  ⚠️  No se detectó columna temporal. Especifica VAR_TIEMPO_OVERRIDE.")
    return None, df_raw


if VAR_TIEMPO_OVERRIDE is not None:
    VAR_TIEMPO = VAR_TIEMPO_OVERRIDE
    print(f"✓ VAR_TIEMPO forzado manualmente: {VAR_TIEMPO!r}")
else:
    VAR_TIEMPO, _df_raw = _detectar_var_tiempo(_df_raw)
    if VAR_TIEMPO:
        print(f"✓ VAR_TIEMPO detectado: {VAR_TIEMPO!r}")

# ── Paso 1: heurística ────────────────────────────────────────────────────
_NO_METRICA = {
    'id', 'codigo', 'code', 'cod', 'clave', 'key',
    'nombre', 'name', 'descripcion', 'description', 'label', 'tag',
    'utm', 'lon', 'lat', 'longitud', 'latitud', 'coordenada',
    'coord', 'norte', 'este', 'x', 'y', 'z',
    'distrito', 'zona', 'area', 'region', 'municipio', 'ciudad',
    'sensor', 'estacion', 'punto', 'ubicacion', 'location',
    'tipo', 'type', 'categoria', 'category', 'clase', 'class',
    'flag', 'estado', 'status', 'activo', 'active',
}

_METRICA_POSITIVA = {
    'intensidad', 'ocupacion', 'flujo', 'velocidad', 'volumen', 'caudal',
    'temperatura', 'presion', 'humedad', 'concentracion', 'nivel',
    'consumo', 'potencia', 'energia', 'demanda', 'produccion',
    'ventas', 'precio', 'importe', 'valor', 'medida', 'lectura',
    'indice', 'tasa', 'ratio', 'porcentaje', 'trafico', 'carga',
    'uso', 'rendimiento', 'eficiencia',
}

def _tokenizar(col):
    return set(re.split(r'[_\-\s]+', col.lower()))

def _heuristica(df, var_tiempo):
    N = len(df)
    claras, ambiguas, info = [], [], {}
    for col in df.columns:
        if col == var_tiempo:
            continue
        serie = df[col]
        tokens = _tokenizar(col)
        # Texto
        if serie.dtype == object:
            info[col] = f"texto → descartada"
            continue
        # Lista negra de tokens
        neg = tokens & _NO_METRICA
        if neg:
            info[col] = f"token no-métrica {neg} → descartada"
            continue
        # Estadísticos
        rango = serie.max() - serie.min()
        cv    = serie.std() / (abs(serie.mean()) + 1e-9)
        n_u   = serie.nunique()
        pct_u = n_u / N
        # Constante
        if rango < 1e-6 or cv < 0.01:
            info[col] = f"constante (cv={cv:.4f}) → descartada"
            continue
        # ID
        if pct_u > 0.95:
            info[col] = f"posible ID ({n_u} únicos) → descartada"
            continue
        # Categórica
        if n_u <= 10:
            info[col] = f"categórica ({n_u} valores) → descartada"
            continue
        # Variabilidad temporal
        try:
            tmp = df[[var_tiempo, col]].copy()
            tmp[var_tiempo] = pd.to_datetime(tmp[var_tiempo])
            tmp['_h'] = tmp[var_tiempo].dt.hour
            ratio_t = tmp.groupby('_h')[col].std().mean() / (serie.std() + 1e-9)
            if ratio_t < 0.05:
                info[col] = f"sin variabilidad temporal → descartada"
                continue
        except:
            pass
        # Clasificar
        pos = tokens & _METRICA_POSITIVA
        if pos and cv > 0.1:
            claras.append(col)
            info[col] = f"✓ CLARA (token={pos}, cv={cv:.2f})"
        else:
            ambiguas.append(col)
            info[col] = f"? AMBIGUA (cv={cv:.2f}, únicos={n_u})"
    return claras, ambiguas, info

_claras, _ambiguas, _info_heuristica = _heuristica(_df_raw, VAR_TIEMPO)

print("DETECCIÓN AUTOMÁTICA DE VARIABLE MÉTRICA")
print("=" * 55)
for col, desc in _info_heuristica.items():
    print(f"  {col:<18} → {desc}")
print()
print(f"  Candidatas claras:   {_claras}")
print(f"  Candidatas ambiguas: {_ambiguas}")
print()

# ── Paso 2: decidir ────────────────────────────────────────────────────────
# Reglas de decisión (sin llamada a API externa):
#   - Override manual      → usar lo que dijo el usuario
#   - 0 candidatas         → avisar, pedir override
#   - 1 candidata total    → automático, sin ambigüedad
#   - >1 CLARAS, 0 AMBIGUAS→ son todas métricas válidas del dataset;
#                            usar la primera y mostrar las demás como opciones
#   - hay AMBIGUAS         → no se puede decidir automáticamente, pedir override

_todas_candidatas = _claras + _ambiguas

if VAR_METRICA_OVERRIDE is not None:
    VAR_METRICA = VAR_METRICA_OVERRIDE
    print(f"✓ Override manual: VAR_METRICA = {VAR_METRICA!r}")

elif len(_todas_candidatas) == 0:
    print("⚠️  No se detectó ninguna métrica candidata.")
    print("    Especifica manualmente: VAR_METRICA_OVERRIDE = 'nombre_columna'")
    VAR_METRICA = None

elif len(_todas_candidatas) == 1:
    VAR_METRICA = _todas_candidatas[0]
    print(f"✓ Detección automática unívoca: VAR_METRICA = {VAR_METRICA!r}")

elif len(_ambiguas) == 0:
    # Varias CLARAS: todas son métricas reales del dataset (ej: intensidad + ocupacion)
    # No hay ambigüedad semántica — el usuario decide cuál analizar en esta ejecución
    VAR_METRICA = _claras[0]
    print(f"✓ Varias métricas detectadas: {_claras}")
    print(f"  Usando: {VAR_METRICA!r}")
    if len(_claras) > 1:
        otras = _claras[1:]
        print(f"  Para analizar otra métrica, pon: VAR_METRICA_OVERRIDE = {otras[0]!r}")

else:
    # Hay AMBIGUAS y la heurística no puede decidir.
    print(f"⚠️  Columnas con nombre incierto: {_ambiguas}")
    print(f"    Métricas claras: {_claras}")

    _metricas_llm = None
    if USAR_LLM_FALLBACK and not _claras:
        # Solo recurrimos al LLM si NO hay ninguna clara: último recurso real.
        print("    → Intentando desambiguar con el LLM...")
        _metricas_llm = _detectar_metrica_via_llm(
            _df_raw, VAR_TIEMPO, _todas_candidatas)

    if _metricas_llm:
        VAR_METRICA = _metricas_llm[0]
        print(f"✓ Métrica seleccionada por LLM: VAR_METRICA = {VAR_METRICA!r}")
        if len(_metricas_llm) > 1:
            print(f"  Otras métricas detectadas: {_metricas_llm[1:]}")
            print(f"  Para analizar otra: VAR_METRICA_OVERRIDE = {_metricas_llm[1]!r}")
    else:
        # Fallback manual (comportamiento original, intacto)
        print(f"    Especifica cuál usar: VAR_METRICA_OVERRIDE = 'nombre_columna'")
        print(f"    Opciones disponibles: {_todas_candidatas}")
        VAR_METRICA = _claras[0] if _claras else _ambiguas[0]
        print(f"    Usando por defecto: {VAR_METRICA!r}")

# ── Paso 3: cargar df con la métrica detectada ────────────────────────────
if VAR_METRICA:
    df = _df_raw[[VAR_TIEMPO, VAR_METRICA]].copy()
    df[VAR_TIEMPO] = pd.to_datetime(df[VAR_TIEMPO])
    df = df.sort_values(VAR_TIEMPO).reset_index(drop=True)
    t0 = df[VAR_TIEMPO].min()
    df["segundos"] = (df[VAR_TIEMPO] - t0).dt.total_seconds().astype(int)
    x     = df["segundos"].to_numpy()
    x_max = int(df["segundos"].max())
    anio_inicio = df[VAR_TIEMPO].min().year
    anio_fin    = df[VAR_TIEMPO].max().year

    _diffs_tmp = df[VAR_TIEMPO].diff().dt.total_seconds().dropna()
    GRANULARIDAD_S = float(_diffs_tmp.median()) if len(_diffs_tmp) else 3600.0
    del _diffs_tmp

    print()
    print(f"Dataset cargado: {len(df):,} filas | métrica: {VAR_METRICA!r} | granularidad: {GRANULARIDAD_S:.0f}s")
    print(df.head(3).to_string(index=False))


In [ ]:
# Función trapecio + tolerancias

# ───── Tolerancia por defecto ─────
# Se aplica a cualquier bloque temporal cuya tolerancia específica sea None.
TOLERANCIA = 0.2  # @param {"type":"number"}

# ───── Tolerancias específicas por bloque temporal ─────
# La tolerancia debería ser proporcional a la incertidumbre
# semántica del concepto, no únicamente a su duración. Un 20% sobre una hora
# (12 min de rampa) genera un solapamiento excesivo entre horas vecinas;
# un 20% sobre un año (~73 días) es razonable para la transición entre años.
TOL_ANIOS      = None   # @param {"type":"raw"}
TOL_MESES      = None   # @param {"type":"raw"}
TOL_SEMANAS    = 0.05   # @param {"type":"raw"}
TOL_HORAS      = 0.5   # @param {"type":"raw"}
TOL_QUINCENAS  = None   # @param {"type":"raw"}
TOL_ESTACIONES = None   # @param {"type":"raw"}
TOL_ABSOLUTAS  = None   # @param {"type":"raw"}

# ───── Muestras mínimas en la rampa ─────
# Garantiza que al menos N muestras del dataset caigan dentro de cada rampa.
# Sin esto, cuando la granularidad ≥ duración del bloque (p. ej. horas con
# datos horarios), la rampa es más estrecha que el intervalo entre muestras
# y la variable acaba siendo crisp (0/1) en lugar de difusa.
# N=2 produce una transición suave con al menos 2 valores intermedios.
N_MUESTRAS_RAMPA = 3  # @param {"type":"integer"}

def tol(especifica):
    """Devuelve la tolerancia específica si está definida; si no, la por defecto."""
    return especifica if especifica is not None else TOLERANCIA

def rampa_s(tolerancia_prop, duracion_bloque):
    """
    Calcula la rampa en SEGUNDOS para un bloque temporal.

    Toma el máximo entre:
      1) la rampa proporcional (tolerancia * duración del bloque), y
      2) un suelo mínimo de N_MUESTRAS_RAMPA * GRANULARIDAD_S.

    Así se garantiza que siempre haya muestras dentro de la rampa,
    independientemente de la relación entre granularidad y duración del bloque.
    """
    rampa_proporcional = tolerancia_prop * duracion_bloque
    rampa_minima       = N_MUESTRAS_RAMPA * GRANULARIDAD_S
    return max(rampa_proporcional, rampa_minima)

def trapecio(x, a, b, c, d):
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)

    m1 = (x >= a) & (x < b)
    y[m1] = (x[m1] - a) / (b - a) if b != a else 1.0

    m2 = (x >= b) & (x <= c)
    y[m2] = 1.0

    m3 = (x > c) & (x <= d)
    y[m3] = (d - x[m3]) / (d - c) if d != c else 1.0

    return np.clip(y, 0, 1).round(4)


print(f"Tolerancia por defecto: {TOLERANCIA}")
print(f"  Años:            {tol(TOL_ANIOS)}")
print(f"  Meses:           {tol(TOL_MESES)}")
print(f"  Semanas (días):  {tol(TOL_SEMANAS)}")
print(f"  Horas:           {tol(TOL_HORAS)}")
print(f"  Quincenas:       {tol(TOL_QUINCENAS)}")
print(f"  Estaciones:      {tol(TOL_ESTACIONES)}")
print(f"  Valores absol.:  {tol(TOL_ABSOLUTAS)}")


# Detección automática de granularidad y activación de bloques


In [ ]:
# Activación de bloques temporales (adaptativo según granularidad)

# ───── Control de activación de bloques ─────
# Cada flag puede valer:
#   - None        → el sistema decide automáticamente según la granularidad y
#                   cobertura del dataset (recomendado).
#   - True / False → el usuario fuerza la activación/desactivación del bloque.
#
# Justificación: un dataset diario no necesita variables horarias; uno de dos
# meses no necesita variables anuales. Generar bloques sin sentido produce
# variables constantes a 0 que ensucian la minería de reglas en src02.
GEN_ANIOS       = None   # @param {"type":"raw"}
GEN_MESES       = None   # @param {"type":"raw"}
GEN_ESTACIONES  = None   # @param {"type":"raw"}
GEN_QUINCENAS   = None   # @param {"type":"raw"}
GEN_DIAS        = None   # @param {"type":"raw"}   # lunes..domingo
GEN_LABORABLES  = None   # @param {"type":"raw"}   # laborable / fin de semana
GEN_FRANJAS     = None   # @param {"type":"raw"}   # madrugada, mañana, tarde, noche
GEN_HORAS       = None   # @param {"type":"raw"}   # 00h..23h
GEN_MINUTOS     = None   # @param {"type":"raw"}   # cuartos de hora dentro de la hora
GEN_MIN_FINOS   = None   # @param {"type":"raw"}   # minutos individuales (solo si gran < 60s)
GEN_FESTIVOS    = None   # @param {"type":"raw"}   # días festivos (siempre útil si hay datos ≥1 día)

# ───── Cobertura del dataset ─────
# (GRANULARIDAD_S ya calculada en la celda de carga)
_t_min, _t_max = df[VAR_TIEMPO].min(), df[VAR_TIEMPO].max()
COBERTURA_S     = (_t_max - _t_min).total_seconds()
COBERTURA_DIAS  = COBERTURA_S / 86400
COBERTURA_MESES = COBERTURA_DIAS / 30.44       # aproximado
COBERTURA_ANIOS = COBERTURA_DIAS / 365.25      # aproximado
N_ANIOS_DISTINTOS = df[VAR_TIEMPO].dt.year.nunique()
N_MESES_DISTINTOS = df[VAR_TIEMPO].dt.to_period("M").nunique()

# Clasificación textual (informativa)
if   GRANULARIDAD_S <= 60:       _nivel = "subminuto"
elif GRANULARIDAD_S <= 3600:     _nivel = "subhora"
elif GRANULARIDAD_S <= 86400:    _nivel = "subdia"
elif GRANULARIDAD_S <= 604800:   _nivel = "subsemana"
else:                            _nivel = "grueso"

print("=" * 60)
print("DIAGNÓSTICO DEL DATASET")
print("=" * 60)
print(f"  Granularidad mediana: {GRANULARIDAD_S:>10.1f} s   ({_nivel})")
print(f"  Cobertura temporal:   {COBERTURA_DIAS:>10.1f} días")
print(f"  Años distintos:       {N_ANIOS_DISTINTOS:>10d}")
print(f"  Meses distintos:      {N_MESES_DISTINTOS:>10d}")

# ───── Reglas de activación automática ─────
# Resolución suficiente: el bloque es más grueso que la granularidad del dato.
# Cobertura suficiente: hay al menos N instancias del bloque en el dataset.
_auto = {
    "ANIOS":      N_ANIOS_DISTINTOS >= 2,
    "MESES":      GRANULARIDAD_S <= 86400   and N_MESES_DISTINTOS >= 2,
    "ESTACIONES": GRANULARIDAD_S <= 604800  and COBERTURA_DIAS   >= 90,
    "QUINCENAS":  GRANULARIDAD_S <= 86400   and COBERTURA_DIAS   >= 30,
    "DIAS":       GRANULARIDAD_S <= 86400   and COBERTURA_DIAS   >= 7,
    "LABORABLES": GRANULARIDAD_S <= 86400   and COBERTURA_DIAS   >= 7,
    "FRANJAS":    GRANULARIDAD_S <= 3600    and COBERTURA_DIAS   >= 1,
    "HORAS":      GRANULARIDAD_S <= 3600    and COBERTURA_DIAS   >= 1,
    "MIN_FINOS":  GRANULARIDAD_S < 60       and COBERTURA_S      >= 3600,  # datos de segundos
    "MINUTOS":    GRANULARIDAD_S < 900      and COBERTURA_S      >= 3600,  # <15 min y ≥1 h
    "FESTIVOS":   GRANULARIDAD_S <= 86400   and COBERTURA_DIAS   >= 1,
}

def _resolver(flag_usuario, clave_auto):
    """None → decisión automática; True/False → el usuario fuerza."""
    return _auto[clave_auto] if flag_usuario is None else bool(flag_usuario)

GEN_ANIOS      = _resolver(GEN_ANIOS,      "ANIOS")
GEN_MESES      = _resolver(GEN_MESES,      "MESES")
GEN_ESTACIONES = _resolver(GEN_ESTACIONES, "ESTACIONES")
GEN_QUINCENAS  = _resolver(GEN_QUINCENAS,  "QUINCENAS")
GEN_DIAS       = _resolver(GEN_DIAS,       "DIAS")
GEN_LABORABLES = _resolver(GEN_LABORABLES, "LABORABLES")
GEN_FRANJAS    = _resolver(GEN_FRANJAS,    "FRANJAS")
GEN_HORAS      = _resolver(GEN_HORAS,      "HORAS")
GEN_MINUTOS    = _resolver(GEN_MINUTOS,    "MINUTOS")
GEN_MIN_FINOS  = _resolver(GEN_MIN_FINOS,  "MIN_FINOS") if "MIN_FINOS" in _auto else False
GEN_FESTIVOS   = _resolver(GEN_FESTIVOS,   "FESTIVOS")

print()
print("BLOQUES TEMPORALES A GENERAR")
print("-" * 60)
for nombre, valor in [
    ("Años",              GEN_ANIOS),
    ("Meses",             GEN_MESES),
    ("Estaciones",        GEN_ESTACIONES),
    ("Quincenas",         GEN_QUINCENAS),
    ("Días (Lun..Dom)",   GEN_DIAS),
    ("Laborable/FinSem",  GEN_LABORABLES),
    ("Franjas del día",   GEN_FRANJAS),
    ("Horas",             GEN_HORAS),
    ("Minutos (cuartos)", GEN_MINUTOS),
    ("Minutos finos (<60s)", GEN_MIN_FINOS),
    ("Festivos",          GEN_FESTIVOS),
]:
    marca = "✓" if valor else "✗"
    print(f"  {marca} {nombre}")
print("=" * 60)


# GENERALES - vTiempo

##Año

In [ ]:
if GEN_ANIOS:
    # BLOQUE 1: AÑO (t_2024, t_2025, ...)
    # =============================================================
    # Una columna difusa por cada año que cubre el dataset.
    # Núcleo (pertenencia=1): todo el año completo (1 ene → 31 dic)
    # Columnas: t_2024, t_2025, ... (automático según los datos)


    print("-- AÑOS --")

    for anio in range(anio_inicio, anio_fin + 1):

        # Inicio y fin del año como timestamps
        inicio_anio = pd.Timestamp(year=anio,     month=1, day=1)
        fin_anio    = pd.Timestamp(year=anio + 1, month=1, day=1)

        # Convertir a segundos relativos al t0 del dataset
        b = (inicio_anio - t0).total_seconds()
        c = (fin_anio    - t0).total_seconds()

        # Rampas de entrada y salida con tolerancia de AÑOS sobre su duración
        duracion_anio = c - b
        _rampa = rampa_s(tol(TOL_ANIOS), duracion_anio)
        a = b - _rampa
        d = c + _rampa

        col_difusa = trapecio(x, a, b, c, d)

        nombre_col = f"t_{anio}"
        df[nombre_col] = col_difusa

        print(f"  {nombre_col}: max={df[nombre_col].max():.2f}  "
              f"registros con valor>0: {(df[nombre_col]>0).sum():,}")

    df

else:
    print("[SKIP] Bloque 'AÑOS' desactivado (GEN_ANIOS=False)")


##Meses

In [ ]:
if GEN_MESES:
    # BLOQUE 2: MESES (t_Ene, t_Feb, ...)
    # =============================================================


    nombres_meses = ["Ene","Feb","Marz","Abr","May","Jun",
                     "Jul","Ago","Sep","Oct","Nov","Dic"]



    print("**MESES**")
    for num_mes, nombre_mes in enumerate(nombres_meses, start=1):
        col_difusa = np.zeros_like(x, dtype=float)

        # Iterar sobre todos los años que cubre el dataset
        for anio in range(anio_inicio, anio_fin + 1):

            # Inicio y fin del mes como timestamps
            inicio_mes = pd.Timestamp(year=anio, month=num_mes, day=1)
            if num_mes == 12:
                fin_mes = pd.Timestamp(year=anio + 1, month=1, day=1)
            else:
                fin_mes = pd.Timestamp(year=anio, month=num_mes + 1, day=1)

            # Convertir a segundos relativos al t0 del dataset
            b = (inicio_mes - t0).total_seconds()
            c = (fin_mes    - t0).total_seconds()

            # Si el mes está completamente fuera del rango, saltar
            if c < 0 or b > x_max:
                continue

            duracion_mes = c - b
            _rampa = rampa_s(tol(TOL_MESES), duracion_mes)
            a = b - _rampa
            d = c + _rampa

            col_difusa = np.maximum(col_difusa, trapecio(x, a, b, c, d))

        nombre_col = f"t_{nombre_mes}"
        df[nombre_col] = col_difusa

    df
else:
    print("[SKIP] Bloque 'MESES' desactivado (GEN_MESES=False)")


## Días de la Semana

In [ ]:
if GEN_DIAS:
    # BLOQUE 3: DÍAS DE LA SEMANA (lunes a domingo)
    # =============================================================
    indice_lunes = df[df[VAR_TIEMPO].dt.weekday == 0].index[0]
    segundos_lunes = df.loc[indice_lunes, "segundos"]

    b0 = int(df.loc[indice_lunes, "segundos"])

    duracion_dia = 24 * 3600
    duracion_semana = 7 * duracion_dia

    x = df["segundos"].to_numpy()
    x_max = int(df["segundos"].max())

    nombres_dias = ["Lun","Mar","Mie","Jue","Vie","Sab","Dom"]

    for n_dia, nombre in enumerate(nombres_dias):

        dia_difuso = np.zeros_like(x, dtype=float)
        offset_dia = n_dia * duracion_dia

        k = 0
        while True:
            b = b0 + offset_dia + k * duracion_semana
            if b > x_max + duracion_semana:
                break

            c = b + duracion_dia
            _rampa = rampa_s(tol(TOL_SEMANAS), duracion_dia)
            a = b - _rampa
            d = c + _rampa

            dia_difuso = np.maximum(dia_difuso, trapecio(x, a, b, c, d))
            k += 1

        df[f"t_{nombre}"] = dia_difuso

    df
else:
    print("[SKIP] Bloque 'DÍAS (Lun..Dom)' desactivado (GEN_DIAS=False)")


##Hora del día

In [ ]:
if GEN_HORAS:
    # BLOQUE 4: HORAS DEL DÍA (hora a hora, 0h–23h)
    # =============================================================
    duracion_hora = 3600
    duracion_dia  = 24 * 3600

    x     = df["segundos"].to_numpy()
    x_max = int(df["segundos"].max())

    # Punto de referencia: primer 00:00 del dataset
    mascara_medianoche = (df[VAR_TIEMPO].dt.hour == 0) & (df[VAR_TIEMPO].dt.minute == 0)
    if mascara_medianoche.any():
        b0_hora = int(df.loc[mascara_medianoche.idxmax(), "segundos"])
    else:
        primer_dia = df[VAR_TIEMPO].min().normalize() + pd.Timedelta(days=1)
        b0_hora = int((primer_dia - t0).total_seconds())

    print("**HORAS DEL DÍA**")

    for hora in range(24):
        col_difusa = np.zeros_like(x, dtype=float)
        offset_hora = hora * duracion_hora

        k = 0
        while True:
            b = b0_hora + offset_hora + k * duracion_dia
            if b > x_max + duracion_dia:
                break
            c = b + duracion_hora
            _rampa = rampa_s(tol(TOL_HORAS), duracion_hora)
            a = b - _rampa
            d = c + _rampa
            col_difusa = np.maximum(col_difusa, trapecio(x, a, b, c, d))
            k += 1

        nombre_col = f"t_H{hora:02d}"
        df[nombre_col] = col_difusa


    print()
    df
else:
    print("[SKIP] Bloque 'HORAS' desactivado (GEN_HORAS=False)")


# ESPECÍFICAS - vTiempo

## Día laborable/Fin de semana


In [ ]:
if GEN_LABORABLES:
    # BLOQUE 5: DÍAS LABORABLES vs FIN DE SEMANA
    # =============================================================
    # tLaborable : lunes a viernes (tLun + tMar + tMie + tJue + tVie)
    # tFinSemana : sábado y domingo (tSab + tDom)


    print("**LABORABLES / FIN DE SEMANA**")

    df["t_Laborable"] = df[["t_Lun","t_Mar","t_Mie","t_Jue","t_Vie"]].max(axis=1).round(4)
    df["t_FinSemana"] = df[["t_Sab","t_Dom"]].max(axis=1).round(4)

    for col in ["t_Laborable", "t_FinSemana"]:
        print(f"  {col}: max={df[col].max():.2f}  "
              f"registros con valor>0: {(df[col]>0).sum():,}")
    df

else:
    print("[SKIP] Bloque 'LABORABLE/FINSEM' desactivado (GEN_LABORABLES=False)")


## DIAS FESTIVOS


In [ ]:
if GEN_FESTIVOS:
    # BLOQUE 10: DÍAS FESTIVOS
    # =============================================================
    # t_Festivo : 1.0 si el día es festivo oficial, 0.0 si no.
    #
    # Usa la librería `holidays` para obtener los festivos nacionales
    # y de la comunidad autónoma especificada.
    #
    # Justificación: un lunes festivo se comporta como un domingo en
    # cuanto a patrones de tráfico. Sin esta variable, el sistema
    # confunde el comportamiento festivo con el comportamiento laboral
    # del mismo día de la semana, degradando la calidad de las reglas.
    #
    # PAIS_FESTIVOS  : código ISO del país (por defecto "ES" = España)
    # SUBDIV_FESTIVOS: subdivisión (comunidad autónoma, estado, etc.)
    #                  Para España: "MD"=Madrid, "CT"=Cataluña, "PV"=País Vasco...
    #                  None = solo festivos nacionales.

    PAIS_FESTIVOS   = "ES"    # @param {"type":"string"}
    SUBDIV_FESTIVOS = "MD"    # @param {"type":"string"}

    try:
        import holidays as hol_lib
        anios_datos = list(range(anio_inicio, anio_fin + 1))
        if SUBDIV_FESTIVOS and SUBDIV_FESTIVOS.strip():
            festivos_set = set(hol_lib.country_holidays(
                PAIS_FESTIVOS, subdiv=SUBDIV_FESTIVOS, years=anios_datos
            ).keys())
        else:
            festivos_set = set(hol_lib.country_holidays(
                PAIS_FESTIVOS, years=anios_datos
            ).keys())

        df["t_Festivo"] = (
            df[VAR_TIEMPO].dt.date.apply(lambda d: 1.0 if d in festivos_set else 0.0)
        )

        n_festivos_dias = df[df["t_Festivo"] == 1.0][VAR_TIEMPO].dt.date.nunique()
        n_festivos_reg  = (df["t_Festivo"] == 1.0).sum()
        print(f"  t_Festivo: {n_festivos_dias} días festivos detectados "
              f"({n_festivos_reg:,} registros con valor 1.0)")
        print(f"  País: {PAIS_FESTIVOS}  Subdivisión: {SUBDIV_FESTIVOS or 'nacional'}")

    except ImportError:
        print("  ⚠️  Librería 'holidays' no disponible. Instala con: pip install holidays")
        print("      Generando t_Festivo=0 para todos los registros.")
        df["t_Festivo"] = 0.0
    except Exception as e:
        print(f"  ⚠️  Error al obtener festivos: {e}")
        print("      Revisa PAIS_FESTIVOS y SUBDIV_FESTIVOS.")
        df["t_Festivo"] = 0.0
else:
    print("[SKIP] Bloque 'FESTIVOS' desactivado (GEN_FESTIVOS=False)")


##Primera/segunda quincena del mes

In [ ]:
if GEN_QUINCENAS:
    # BLOQUE 6: PRIMERA / SEGUNDA QUINCENA DEL MES
    # =============================================================
    # tQ1mes : días 1–15 de cada mes
    # tQ2mes : días 16–fin de mes


    print("**QUINCENAS**")

    duracion_quincena = 15 * 24 * 3600   # 15 días en segundos (aproximación)

    for nombre_col, dia_inicio, dia_fin_offset in [
        ("t_Q1mes", 1,  15),   # del día 1 al 15
        ("t_Q2mes", 16, None), # del día 16 al último del mes
    ]:
        col_difusa = np.zeros_like(x, dtype=float)

        for anio in range(anio_inicio, anio_fin + 1):
            for mes in range(1, 13):

                inicio = pd.Timestamp(year=anio, month=mes, day=dia_inicio)

                if dia_fin_offset is None:
                    # Segunda quincena: hasta el último día del mes
                    if mes == 12:
                        fin = pd.Timestamp(year=anio + 1, month=1, day=1)
                    else:
                        fin = pd.Timestamp(year=anio, month=mes + 1, day=1)
                else:
                    # Primera quincena: hasta el día 15 a medianoche
                    fin = pd.Timestamp(year=anio, month=mes, day=dia_fin_offset,
                                       hour=23, minute=59, second=59)

                b = (inicio - t0).total_seconds()
                c = (fin    - t0).total_seconds()

                if c < 0 or b > x_max:
                    continue

                duracion = c - b
                _rampa = rampa_s(tol(TOL_QUINCENAS), duracion)
                a = b - _rampa
                d = c + _rampa

                col_difusa = np.maximum(col_difusa, trapecio(x, a, b, c, d))

        df[nombre_col] = col_difusa
        print(f"  {nombre_col}: max={df[nombre_col].max():.2f}  "
              f"registros con valor>0: {(df[nombre_col]>0).sum():,}")
    df

else:
    print("[SKIP] Bloque 'QUINCENAS' desactivado (GEN_QUINCENAS=False)")


##Estaciónes del año

In [ ]:
if GEN_ESTACIONES:
    # BLOQUE 7: ESTACIONES DEL AÑO
    # =============================================================
    # Estaciones meteorológicas (hemisferio norte):
    #   tPrimavera : 20 marz – 20 jun
    #   tVerano    : 21 jun – 22 sep
    #   tOtonio    : 23 sep – 20 dic
    #   tInvierno  : 21 dic – 19 marz (cruza año)


    print("**ESTACIONES**")

    estaciones = {
        "t_Primavera": ((3, 20), (6, 20)),
        "t_Verano":    ((6, 21), (9, 22)),
        "t_Otonio":    ((9, 23), (12, 20)),
    }

    # Primavera, Verano y Otoño (no cruzan el año)
    for nombre_col, ((m_ini, d_ini), (m_fin, d_fin)) in estaciones.items():
        col_difusa = np.zeros_like(x, dtype=float)

        for anio in range(anio_inicio, anio_fin + 1):
            inicio = pd.Timestamp(year=anio, month=m_ini, day=d_ini)
            fin    = pd.Timestamp(year=anio, month=m_fin, day=d_fin)

            b = (inicio - t0).total_seconds()
            c = (fin    - t0).total_seconds()

            if c < 0 or b > x_max:
                continue

            duracion = c - b
            _rampa = rampa_s(tol(TOL_ESTACIONES), duracion)
            a = b - _rampa
            d_trap = c + _rampa

            col_difusa = np.maximum(col_difusa, trapecio(x, a, b, c, d_trap))

        df[nombre_col] = col_difusa
        print(f"  {nombre_col}: max={df[nombre_col].max():.2f}  "
              f"registros con valor>0: {(df[nombre_col]>0).sum():,}")

    # Invierno (cruza el año: 21 dic → 19 marz del año siguiente)
    col_invierno = np.zeros_like(x, dtype=float)
    for anio in range(anio_inicio - 1, anio_fin + 1):
        inicio = pd.Timestamp(year=anio,     month=12, day=21)
        fin    = pd.Timestamp(year=anio + 1, month=3,  day=19)

        b = (inicio - t0).total_seconds()
        c = (fin    - t0).total_seconds()

        if c < 0 or b > x_max:
            continue

        duracion = c - b
        _rampa = rampa_s(tol(TOL_ESTACIONES), duracion)
        a = b - _rampa
        d_trap = c + _rampa

        col_invierno = np.maximum(col_invierno, trapecio(x, a, b, c, d_trap))

    df["t_Invierno"] = col_invierno
    print(f"  t_Invierno:   max={df['t_Invierno'].max():.2f}  "
          f"registros con valor>0: {(df['t_Invierno']>0).sum():,}")
    df

else:
    print("[SKIP] Bloque 'ESTACIONES' desactivado (GEN_ESTACIONES=False)")


## Franjas del día

In [ ]:
if GEN_FRANJAS:
    # BLOQUE 8: FRANJAS DEL DÍA
    # =============================================================
    #   tMadrugada : 00h – 06h
    #   tMañana    : 07h – 13h
    #   tTarde     : 14h – 20h
    #   tNoche     : 21h – 23h  (cruza medianoche hacia la madrugada)


    print("**FRANJAS DEL DÍA**")

    franjas = {
        "t_Madrugada": [f"t_H{h:02d}" for h in range(0,  7)],   # 00–06
        "t_Mañana":    [f"t_H{h:02d}" for h in range(7,  14)],  # 07–13
        "t_Tarde":     [f"t_H{h:02d}" for h in range(14, 21)],  # 14–20
        "t_Noche":     [f"t_H{h:02d}" for h in range(21, 24)],  # 21–23
    }

    for nombre_franja, cols_horas in franjas.items():
        existing_cols = [col for col in cols_horas if col in df.columns]
        if existing_cols:
            df[nombre_franja] = df[existing_cols].max(axis=1).round(4)
            print(f"  {nombre_franja}: max={df[nombre_franja].max():.2f}  " +
                  f"registros con valor>0: {(df[nombre_franja]>0).sum():,}")
        else:
            df[nombre_franja] = 0.0
            print(f"  {nombre_franja}: No se encontraron columnas de horas correspondientes.")

    df
else:
    print("[SKIP] Bloque 'FRANJAS DEL DÍA' desactivado (GEN_FRANJAS=False)")


## Minutos (caurtos de hora)

In [ ]:
if GEN_MINUTOS:
    # BLOQUE 9: MINUTOS DEL RELOJ (cuartos de hora)
    # =============================================================
    # Genera 4 variables difusas que representan los cuartos de hora
    # dentro de cualquier hora del reloj:
    #   t_M00 : minutos 00–14 (primer cuarto)
    #   t_M15 : minutos 15–29 (segundo cuarto)
    #   t_M30 : minutos 30–44 (tercer cuarto)
    #   t_M45 : minutos 45–59 (cuarto cuarto)
    #
    # Se activa automáticamente solo si la granularidad ≤ 15 min.
    # Granularidad más fina que cuartos se agrega a este mismo esquema
    # para evitar la explosión combinatoria de 60 variables de minuto.

    duracion_cuarto = 15 * 60        # 900 s
    duracion_hora_s = 3600

    x     = df["segundos"].to_numpy()
    x_max = int(df["segundos"].max())

    # Punto de referencia: primer instante con minuto=0 y segundo=0
    mascara_mm00 = (df[VAR_TIEMPO].dt.minute == 0) & (df[VAR_TIEMPO].dt.second == 0)
    if mascara_mm00.any():
        b0_min = int(df.loc[mascara_mm00.idxmax(), "segundos"])
    else:
        primer_h = df[VAR_TIEMPO].min().floor("H") + pd.Timedelta(hours=1)
        b0_min   = int((primer_h - t0).total_seconds())

    print("**MINUTOS (CUARTOS DE HORA)**")

    for cuarto_idx, minuto_inicio in enumerate([0, 15, 30, 45]):
        col_difusa  = np.zeros_like(x, dtype=float)
        offset_min  = minuto_inicio * 60

        k = 0
        while True:
            b = b0_min + offset_min + k * duracion_hora_s
            if b > x_max + duracion_hora_s:
                break
            c = b + duracion_cuarto
            _rampa = rampa_s(tol(TOL_HORAS), duracion_cuarto)
            a = b - _rampa
            d = c + _rampa
            col_difusa = np.maximum(col_difusa, trapecio(x, a, b, c, d))
            k += 1

        nombre_col = f"t_M{minuto_inicio:02d}"
        df[nombre_col] = col_difusa
        print(f"  {nombre_col}: max={df[nombre_col].max():.2f}  "
              f"registros con valor>0: {(df[nombre_col]>0).sum():,}")
else:
    print("[SKIP] Bloque 'MINUTOS' desactivado (GEN_MINUTOS=False)")


#GENERALES - vMetrica

In [ ]:
# 1. Cálculo de estadísticos
max_val = df[VAR_METRICA].max()
p10 = df[VAR_METRICA].quantile(0.10)
p25 = df[VAR_METRICA].quantile(0.25)
p35 = df[VAR_METRICA].quantile(0.35)
p45 = df[VAR_METRICA].quantile(0.45)
p55 = df[VAR_METRICA].quantile(0.55)
p65 = df[VAR_METRICA].quantile(0.65)
p75 = df[VAR_METRICA].quantile(0.75)
p90 = df[VAR_METRICA].quantile(0.90)

# 2. Generación de las 5 Variables Difusas (Forma Trapezoidal)
# Cada trapecio se define como: (inicio_subida, inicio_núcleo, fin_núcleo, fin_bajada)

# MUY BAJA: Estable en 1 desde el inicio hasta P10, cae hacia P25
min_val = df[VAR_METRICA].min()
df["v_MuyBaja"] = trapecio(df[VAR_METRICA], min_val - 1, min_val, p10, p25)

# BAJA: Sube desde P10, estable entre P25 y P35, cae hacia P50
df["v_Baja"] = trapecio(df[VAR_METRICA], p10, p25, p35, p45)

# MEDIA: Sube desde P35, estable entre P45 y P55, cae hacia P65
# (Usamos P45 y P55 para que el 1.0 no sea solo un punto en el P50)
df["v_Media"] = trapecio(df[VAR_METRICA], p35, p45, p55, p65)

# ALTA: Sube desde P50, estable entre P65 y P75, cae hacia P90
df["v_Alta"] = trapecio(df[VAR_METRICA], p55, p65, p75, p90)

# MUY ALTA: Sube desde P75, estable desde P90 hasta el máximo
df["v_MuyAlta"] = trapecio(df[VAR_METRICA], p75, p90, max_val, max_val + 1)

# 3. Verificación
cols_metrica = ["v_MuyBaja", "v_Baja", "v_Media", "v_Alta", "v_MuyAlta"]
for col in cols_metrica:
    print(f"{col}: registros con pertenencia > 0.5: {(df[col] > 0.5).sum():,}")

In [ ]:

# Ordenar por intensidad para la gráfica
df_plot = df.sort_values(VAR_METRICA)

plt.figure(figsize=(10, 5))
plt.plot(df_plot[VAR_METRICA], df_plot["v_MuyBaja"], label="Muy Baja")
plt.plot(df_plot[VAR_METRICA], df_plot["v_Baja"], label="Baja")
plt.plot(df_plot[VAR_METRICA], df_plot["v_Media"], label="Media")
plt.plot(df_plot[VAR_METRICA], df_plot["v_Alta"], label="Alta")
plt.plot(df_plot[VAR_METRICA], df_plot["v_MuyAlta"], label="Muy Alta")
plt.title(f"Funciones de Pertenencia — {VAR_METRICA}")
plt.xlabel(f"Valor de {VAR_METRICA}")
plt.ylabel("Grado de pertenencia")
plt.legend()
plt.show()

In [ ]:
# BLOQUE MÉTRICA - PARTE 1: Variable difusa en la MEDIANA (P50)
# =============================================================
p50 = df[VAR_METRICA].quantile(0.50)
p25 = df[VAR_METRICA].quantile(0.25)
p75 = df[VAR_METRICA].quantile(0.75)

# Núcleo: la mediana. Rampas: P25 (subida) y P75 (bajada)
df["v_Mediana"] = trapecio(df[VAR_METRICA], p25, p50, p50, p75)

print(f"P25:{p25:.2f}  P50:{p50:.2f}  P75:{p75:.2f}")
print(f"v_Mediana: registros con pertenencia > 0.5: {(df['v_Mediana'] > 0.5).sum():,}")

In [ ]:
# BLOQUE MÉTRICA - PARTE 2: Valores fuera de media ± 2·std (outliers)
# =============================================================
mean_val = df[VAR_METRICA].mean()
std_val  = df[VAR_METRICA].std()
min_val  = df[VAR_METRICA].min()
max_val  = df[VAR_METRICA].max()

m_minus_2std = mean_val - 2 * std_val
m_plus_2std  = mean_val + 2 * std_val

# Outlier bajo: valores por debajo de media - 2·std
# Núcleo: desde el mínimo hasta media-2std, rampa de salida hacia media-std
df["v_OutlierBajo"] = trapecio(
    df[VAR_METRICA],
    min_val - 1,       # a: antes del mínimo (pertenencia ya = 1 desde el inicio)
    min_val,           # b: núcleo empieza en el mínimo real
    m_minus_2std,      # c: núcleo acaba en media - 2std
    mean_val - std_val # d: rampa de bajada hasta media - std
)

# Outlier alto: valores por encima de media + 2·std
# Núcleo: desde media+2std hasta el máximo, rampa de subida desde media+std
df["v_OutlierAlto"] = trapecio(
    df[VAR_METRICA],
    mean_val + std_val, # a: rampa de subida desde media + std
    m_plus_2std,        # b: núcleo empieza en media + 2std
    max_val,            # c: núcleo acaba en el máximo real
    max_val + 1         # d: más allá del máximo (pertenencia = 1 hasta el final)
)

print(f"Umbral bajo: media-2std = {m_minus_2std:.2f}")
print(f"Umbral alto: media+2std = {m_plus_2std:.2f}")
print(f"v_OutlierBajo: registros con pertenencia > 0.5: {(df['v_OutlierBajo'] > 0.5).sum():,}")
print(f"v_OutlierAlto: registros con pertenencia > 0.5: {(df['v_OutlierAlto'] > 0.5).sum():,}")

In [ ]:
# BLOQUE MÉTRICA - PARTE 3: Valores absolutos con escala lógica
# =============================================================

def calcular_breakpoints_logicos(min_v, max_v):
    """
    Dado el rango real de la variable, devuelve una lista de breakpoints
    "redondos" que cubren ese rango de forma lógica y uniforme.

    Ejemplos:
      [0, 10]   → [0, 5, 10]
      [0, 20]   → [0, 5, 10, 15, 20]
      [0, 100]  → [0, 25, 50, 75, 100]
      [0, 1000] → [0, 250, 500, 750, 1000]
      [-10, 10] → [-10, -5, 0, 5, 10]
    """
    rango = max_v - min_v
    if rango == 0:
        return [min_v]

    # Escalones candidatos en orden de preferencia
    escalones_candidatos = [
        0.01, 0.02, 0.025, 0.05,
        0.1, 0.2, 0.25, 0.5,
        1, 2, 2.5, 5,
        10, 20, 25, 50,
        100, 200, 250, 500,
        1000, 2000, 2500, 5000
    ]

    # Elegir el escalón que dé entre 3 y 6 intervalos
    escalon = None
    for e in escalones_candidatos:
        n_intervalos = rango / e
        if 3 <= n_intervalos <= 6:
            escalon = e
            break

    # Fallback: dividir en 4 partes iguales redondeadas
    if escalon is None:
        escalon = round(rango / 4, 2)

    # Generar breakpoints desde el primer múltiplo del escalón >= min_v
    primer_bp = np.ceil(min_v / escalon) * escalon
    breakpoints = []
    bp = primer_bp
    while bp <= max_v + escalon * 0.01:
        breakpoints.append(round(bp, 10))
        bp += escalon

    # Asegurar que el mínimo y el máximo están incluidos
    if breakpoints[0] > min_v:
        breakpoints.insert(0, min_v)
    if breakpoints[-1] < max_v:
        distancia_al_penultimo = max_v - breakpoints[-1]
        if distancia_al_penultimo > escalon * 0.3:  # ← solo añadir si está a >30% del escalón
            breakpoints.append(max_v)

    return breakpoints


def generar_variables_absolutas(df, var_metrica, tolerancia=0.2):
    min_v = df[var_metrica].min()
    max_v = df[var_metrica].max()
    breakpoints = calcular_breakpoints_logicos(min_v, max_v)

    print(f"\nRango detectado: [{min_v:.2f}, {max_v:.2f}]")
    print(f"Breakpoints lógicos: {[round(b, 4) for b in breakpoints]}")
    print()

    cols_generadas = []

    for i, bp in enumerate(breakpoints):

        # ── Rampas tipo "partición Ruspini": cada rampa va hasta el centro del vecino ──
        # La rampa de bajada llega hasta el breakpoint vecino (no hasta la mitad).
        # Esto garantiza que la suma de pertenencias sea siempre >= 1 en todo el dominio
        # (estándar en la literatura de lógica difusa).
        if i == 0:
            rampa_izq = (breakpoints[1] - bp)  # llega hasta el siguiente
        else:
            rampa_izq = (bp - breakpoints[i - 1])  # llega hasta el anterior

        if i == len(breakpoints) - 1:
            rampa_der = (bp - breakpoints[i - 1])  # llega hasta el anterior
        else:
            rampa_der = (breakpoints[i + 1] - bp)  # llega hasta el siguiente
        # ─────────────────────────────────────────────────────────────────────

        ancho_nucleo = min(rampa_izq, rampa_der) * 0.3
        a = bp - rampa_izq
        b = bp - ancho_nucleo / 2
        c = bp + ancho_nucleo / 2
        d = bp + rampa_der

        nombre_bp = str(round(bp, 4)).replace(".", "_").replace("-", "neg")
        col = f"v_abs_{nombre_bp}"

        df[col] = trapecio(df[var_metrica], a, b, c, d)
        cols_generadas.append(col)

        print(f"  {col}: núcleo=[{b:.2f}, {c:.2f}]  "
              f"rampas=[{a:.2f}, {d:.2f}]  "
              f"registros con pertenencia > 0.5: {(df[col] > 0.5).sum():,}")

    return cols_generadas


cols_absolutas = generar_variables_absolutas(df, VAR_METRICA, tolerancia=tol(TOL_ABSOLUTAS))

# Mas avances

In [ ]:
print(f"TOL_ABSOLUTAS efectiva: {tol(TOL_ABSOLUTAS)}")
# Si imprime 0.2 → el problema es que la tolerancia controla las rampas
# y 0.2 es insuficiente para garantizar solapamiento

In [ ]:
# Gráfica con todas las variables nuevas + las existentes

df_plot = df.sort_values(VAR_METRICA)

fig, axes = plt.subplots(3, 1, figsize=(11, 13))

# --- Subgráfica 1: originales + mediana ---
ax = axes[0]
for col in ["v_MuyBaja", "v_Baja", "v_Media", "v_Alta", "v_MuyAlta"]:
    ax.plot(df_plot[VAR_METRICA], df_plot[col], label=col)
ax.plot(df_plot[VAR_METRICA], df_plot["v_Mediana"],
        label="v_Mediana", linestyle="--", linewidth=2)
ax.set_title("Variables originales + Mediana")
ax.set_ylabel("Grado de pertenencia")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Subgráfica 2: outliers ---
ax = axes[1]
ax.plot(df_plot[VAR_METRICA], df_plot["v_OutlierBajo"],
        label="v_OutlierBajo", color="steelblue")
ax.plot(df_plot[VAR_METRICA], df_plot["v_OutlierAlto"],
        label="v_OutlierAlto", color="firebrick")
ax.axvline(m_minus_2std, color="steelblue", linestyle=":", alpha=0.6, label=f"media-2std={m_minus_2std:.1f}")
ax.axvline(m_plus_2std,  color="firebrick", linestyle=":", alpha=0.6, label=f"media+2std={m_plus_2std:.1f}")
ax.set_title("Outliers (fuera de media ± 2·std)")
ax.set_ylabel("Grado de pertenencia")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# --- Subgráfica 3: valores absolutos ---
ax = axes[2]
for col in cols_absolutas:
    ax.plot(df_plot[VAR_METRICA], df_plot[col], label=col)
ax.set_title("Variables absolutas (breakpoints lógicos)")
ax.set_xlabel(f"Valor de {VAR_METRICA}")
ax.set_ylabel("Grado de pertenencia")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def filtrar_variables_constantes(df, prefijos=['t_', 'v_']):
    """
    Elimina columnas que empiezan con ciertos prefijos si son constantes (todo 0 o todo 1).
    """
    cols_a_eliminar = []
    cols_difusas = [c for c in df.columns if any(c.startswith(p) for p in prefijos)]

    for col in cols_difusas:
        if df[col].nunique() <= 1:
            cols_a_eliminar.append(col)

    df_filtrado = df.drop(columns=cols_a_eliminar)
    print(f"Columnas eliminadas por ser constantes: {cols_a_eliminar}")
    return df_filtrado
df = filtrar_variables_constantes(df)


In [ ]:
# Guardar DataFrame fuzzificado
# =============================================================
nombre_dataset   = FICHERO.split('/')[-1].replace('.csv', '')
nombre_metrica  = VAR_METRICA
ruta_fuzzy      = f"data/{nombre_dataset}_{nombre_metrica}_fuzzy.csv"

df.to_csv(ruta_fuzzy, index=False)
print(f"DataFrame fuzzificado guardado en: {ruta_fuzzy}")
print(f"Shape: {df.shape}  ({df.shape[0]:,} filas × {df.shape[1]} columnas)")